# 3. Data Wrangling

**Goal:** turn each launch's landing outcome into a training label, `Class`: 1 if the first stage landed and 0 if it did not.

This notebook loads IBM's reference copy of step 1's output, so the results match the course.

In [1]:
import pandas as pd

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


In [2]:
# Percent of missing values in each column
(df.isnull().sum() / len(df) * 100).round(1)

FlightNumber       0.0
Date               0.0
BoosterVersion     0.0
PayloadMass        0.0
Orbit              0.0
LaunchSite         0.0
Outcome            0.0
Flights            0.0
GridFins           0.0
Reused             0.0
Legs               0.0
LandingPad        28.9
Block              0.0
ReusedCount        0.0
Serial             0.0
Longitude          0.0
Latitude           0.0
dtype: float64

## Launches per site, and launches per orbit

In [3]:
df["LaunchSite"].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

In [4]:
df["Orbit"].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

## Landing outcomes

Each outcome has two parts: whether the landing succeeded (`True`, `False` or `None` for no attempt), and where it landed. `ASDS` is a drone ship at sea, `RTLS` is a landing pad on the ground, and `Ocean` is a controlled landing in the ocean.

In [5]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

Every outcome that does not start with `True` is a failed or missing landing.

In [6]:
bad_outcomes = {outcome for outcome in landing_outcomes.keys() if not outcome.startswith("True")}
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

## Create the landing label

In [7]:
df["Class"] = [0 if outcome in bad_outcomes else 1 for outcome in df["Outcome"]]
print(f"Share of first stages that landed: {df['Class'].mean():.1%}")
df[["Outcome", "Class"]].head(8)

Share of first stages that landed: 66.7%


,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1


In [8]:
df.to_csv("../data/dataset_part_2.csv", index=False)